# Procesamiento Fiscal · `Fiscal-<Depto>_AUTO.xlsx` para Loreto, San Martín y Ucayali

Lee todo lo que producen los scrapers en `Fiscal/Scrapeo/output/` más el `IPC.xlsx` de
`Fiscal/Procesamiento/input/`, y genera **un workbook nuevo por departamento** en
`Fiscal/Procesamiento/output/Fiscal-<Depto>_AUTO.xlsx` con la misma estructura que el
workbook manual (series mensuales en miles de soles, columnas Total por año, bloques
reales deflactados con IPC, variación acumulada Ene.-Mes vs año anterior):

| Hoja | Bloques | Input que la alimenta |
|---|---|---|
| `GASTO G.N.` / `GASTOS G.R.` / `GASTOS G.L.` | Gasto por genérica, servicio de la deuda, gasto por fuente, bloque real | `DEV Generica.*`, `DEV Deuda.*`, `DEV Fuente.*` |
| `INVERSIÓN G.N.` / `G.R.` / `G.L.` | Inversión (2.6) por función y por fuente | `DEV Funcion.*`, `DEV InvFuente.*` |
| `PIM G.N.` / `G.R.` / `G.L.` | PIM anual por función, devengado 2.6 acumulado y % de avance | `PIM Funcion*`, `DEV Funcion.*` |
| `CONTROL` | Estado de todos los CSV de control (ESTADO = OK / PARCIAL / REVISAR) | `CARPETA DE CONTROL`, `PIM/*/CONTROL` |

Si falta un input (por ejemplo aún no corriste el scraper de genéricas), el bloque
correspondiente queda vacío y se avisa en el resumen; nada se inventa.

In [ ]:
# ═══ BLOQUE 1 · LIBRERÍAS Y DRIVE ═══
!pip install openpyxl pandas numpy --quiet
import os, re, glob, unicodedata, warnings
from datetime import date, datetime
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter as L
from IPython.display import display
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive")
RUTA_FISCAL = "/content/drive/MyDrive/Fiscal"
assert os.path.isdir(RUTA_FISCAL), f"No encuentro {RUTA_FISCAL}"
print("✅ Drive montado")


In [ ]:
# ═══ BLOQUE 2 · CONFIGURACIÓN ═══
RUTA_SCRAPEO = os.path.join(RUTA_FISCAL, "Scrapeo", "output")
RUTA_IPC     = os.path.join(RUTA_FISCAL, "Procesamiento", "input", "IPC.xlsx")
RUTA_SALIDA  = os.path.join(RUTA_FISCAL, "Procesamiento", "output")
os.makedirs(RUTA_SALIDA, exist_ok=True)

DEPARTAMENTOS = {"16": ("LORETO", "Loreto"), "22": ("SAN MARTÍN", "SanMartin"), "25": ("UCAYALI", "Ucayali")}
AÑO_INICIO = 2025                 # primer año de las series (los scrapers empiezan en 2025)
IPC_BASE_PROM_ANUAL = 2007        # base del deflactor: IPC_t / promedio(IPC año base). El workbook manual usa 2007 (≈63.78)
IPC_BASE_FIJO = 63.78             # se usa si el IPC.xlsx no cubre el año base (empieza en 2010)
SUFIJO_SALIDA = "_AUTO"           # Fiscal-Loreto_AUTO.xlsx (no toca el archivo manual)
AGRUPAR_COMUNICACIONES_EN_TRANSPORTE = True   # convención del workbook manual: 16: Comunicaciones se suma en 15: Transporte

NIVELES = [  # (corto en nombres de archivo, carpeta DEVENGADO, etiqueta PIM, nombre hoja gasto, nombre hoja inversión, sufijo PIM, título)
    dict(corto="GobNacional",  carpeta="Gobierno Nacional", pim="GOBIERNO NACIONAL",    hoja_g="GASTO G.N.",  hoja_i="INVERSIÓN G.N.", hoja_p="PIM G.N.", nombre="GOBIERNO NACIONAL"),
    dict(corto="GobRegional",  carpeta="Gobierno Regional", pim="GOBIERNOS REGIONALES", hoja_g="GASTOS G.R.", hoja_i="INVERSIÓN G.R.", hoja_p="PIM G.R.", nombre="GOBIERNO REGIONAL"),
    dict(corto="Municipalidad",carpeta="Gobierno Municipal",pim="GOBIERNOS LOCALES",    hoja_g="GASTOS G.L.", hoja_i="INVERSIÓN G.L.", hoja_p="PIM G.L.", nombre="GOBIERNOS LOCALES"),
]

# Filas del bloque por genérica (como en el workbook manual). Cada fila se identifica por la ETIQUETA que
# devuelve el MEF (no por el código, que cambia de formato) y por el tipo de transacción: 5 corriente / 6 capital / 7 deuda.
GENERICAS = [
    ("GASTOS CORRIENTES", None),
    ("Personal y obligaciones",               ("5", r"personal")),
    ("Pensiones y prestaciones sociales",     ("5", r"pension")),
    ("Bienes y servicios",                    ("5", r"bienes y servicios")),
    ("Donaciones y transferencias",           ("5", r"donaciones")),
    ("Otros gastos corrientes",               ("5", r"otros gastos")),
    ("GASTOS DE CAPITAL", None),
    ("Donaciones y transferencias (2)",       ("6", r"donaciones")),
    ("Adquisición de activos no financieros", ("6", r"activos no financieros")),
    ("Inversiones financieras",               ("6", r"activos financieros")),
    ("Otros gastos de capital",               ("6", r"otros gastos")),
    ("SERVICIO DE LA DEUDA", None),
    ("Intereses+comisiones",                  "DEUDA:INTERESES Y COMISIONES"),
    ("Amortización",                          "DEUDA:AMORTIZACION"),
]
# Filas del bloque por fuente, identificadas por la etiqueta de la fuente
FUENTES = [
    ("Recursos Ordinarios (Asignación MEF)",      r"recursos ordinarios"),
    ("Recursos determinados",                     r"recursos determinados"),
    ("Recursos Directamente Recaudados",          r"directamente recaudados"),
    ("Recursos por Oper. Ofi. De Créd. Interno",  r"operaciones oficiales de credito"),
    ("Donaciones y Transferencias",               r"donaciones"),
]
def normalizar(txt):
    import unicodedata
    t = unicodedata.normalize("NFKD", str(txt or ""))
    return "".join(c for c in t if not unicodedata.combining(c)).lower().strip()

def clasificar_generica(cod, etiqueta):
    """-> nombre de fila de GENERICAS o None. Tipo de transacción = primer tramo del código ('5-21' -> '5')."""
    tipo, t = str(cod).strip().split("-")[0], normalizar(etiqueta)
    t = re.sub(r"^[\w\-\.]+\s*:\s*", "", t)          # quita el código de la etiqueta
    for nombre, regla in GENERICAS:
        if isinstance(regla, tuple) and regla[0] == tipo and re.search(regla[1], t):
            return nombre
    return None

def clasificar_deuda(ruta):
    """Clasifica una fila del scraper de deuda por el texto de su RUTA (respaldo para CSV con CLASIFICACION = OTRO)."""
    t = normalizar(ruta)
    return "AMORTIZACION" if "amortiz" in t else "INTERESES Y COMISIONES" if ("interes" in t or "comision" in t) else "OTRO"

def clasificar_fuente(etiqueta):
    t = re.sub(r"^[\w\-\.]+\s*:\s*", "", normalizar(etiqueta))
    for nombre, regla in FUENTES:
        if re.search(regla, t):
            return nombre
    return None

FUNCIONES = ["01: LEGISLATIVA", "02: RELACIONES EXTERIORES", "03: PLANEAMIENTO, GESTION Y RESERVA DE CONTINGENCIA",
             "04: DEFENSA Y SEGURIDAD NACIONAL", "05: ORDEN PUBLICO Y SEGURIDAD", "06: JUSTICIA", "07: TRABAJO",
             "08: COMERCIO", "09: TURISMO", "10: AGROPECUARIA", "11: PESCA", "12: ENERGIA", "13: MINERIA",
             "14: INDUSTRIA", "15: TRANSPORTE", "16: COMUNICACIONES", "17: AMBIENTE", "18: SANEAMIENTO",
             "19: VIVIENDA Y DESARROLLO URBANO", "20: SALUD", "21: CULTURA Y DEPORTE", "22: EDUCACION",
             "23: PROTECCION SOCIAL", "24: PREVISION SOCIAL", "25: DEUDA PUBLICA"]


In [ ]:
# ═══ BLOQUE 3 · LECTURA DE INPUTS ═══
_RE_MES = re.compile(r"(\d{4}) - Mes (\d{2}) - ")

def leer_csvs(patron):
    """Concatena todos los CSV que cumplen el patrón (glob). Añade AÑO/MES_NUM desde el nombre si hace falta."""
    archivos = sorted(glob.glob(patron))
    partes = []
    for a in archivos:
        df = pd.read_csv(a, encoding="utf-8-sig")
        m = _RE_MES.search(os.path.basename(a))
        if m:
            df["AÑO"] = int(m.group(1)); df["MES_NUM"] = int(m.group(2))
        elif "MES" in df.columns:
            df["MES_NUM"] = df["MES"].astype(str).str.extract(r"^(\d+)")[0].astype(int)
        df["ARCHIVO"] = os.path.basename(a)
        partes.append(df)
    if not partes:
        return pd.DataFrame()
    df = pd.concat(partes, ignore_index=True)
    df["COD_DEPTO"] = df["DEPARTAMENTO"].astype(str).str.extract(r"^(\d+)")[0]
    if "MES_NUM" in df.columns:
        df["FECHA"] = pd.to_datetime(dict(year=df["AÑO"], month=df["MES_NUM"], day=1))
    return df

def codigo(serie):
    return serie.astype(str).str.extract(r"^([\w\-\.]+)\s*:")[0]

inputs, faltantes = {}, []
for n in NIVELES:
    carpeta = os.path.join(RUTA_SCRAPEO, "DEVENGADO", n["carpeta"])
    for dim in ["Funcion", "Generica", "Fuente", "InvFuente", "Deuda"]:
        df = leer_csvs(os.path.join(carpeta, f"DEV {dim}.{n['corto']} * - Mes * - *.csv"))
        inputs[(n["corto"], dim)] = df
        if df.empty:
            faltantes.append(f"DEV {dim}.{n['corto']}")
    # el scraper municipal trae el detalle por municipalidad: se agrega al departamento
    f = inputs[(n["corto"], "Funcion")]
    if not f.empty:
        f["COD_FUNCION"] = codigo(f["FUNCION"])
        inputs[(n["corto"], "Funcion")] = (f.groupby(["COD_DEPTO", "FECHA", "AÑO", "MES_NUM", "COD_FUNCION", "FUNCION"], as_index=False)["DEVENGADO"].sum())

pim = leer_csvs(os.path.join(RUTA_SCRAPEO, "PIM", "PIM X FUNCIÓN", "DATA", "PIM Funcion * - *.csv"))
if not pim.empty:
    pim["COD_FUNCION"] = codigo(pim["FUNCION"])
else:
    faltantes.append("PIM Funcion")

controles = []
for a in sorted(glob.glob(os.path.join(RUTA_SCRAPEO, "**", "CONTROL*.csv"), recursive=True)):
    c = pd.read_csv(a, encoding="utf-8-sig")
    if "ESTADO" in c.columns:
        controles.append(pd.DataFrame({"ARCHIVO": os.path.basename(a), "AÑO": c.get("AÑO"), "DEPARTAMENTO": c.get("DEPARTAMENTO"),
                                       "ESTADO": c["ESTADO"], "FECHA_EXTRACCION": c.get("FECHA_EXTRACCION")}))
controles = pd.concat(controles, ignore_index=True) if controles else pd.DataFrame()

# IPC (INEI, Lima, Dic.2021=100): columnas Fecha | IPC a partir de la fila 4
ipc_raw = pd.read_excel(RUTA_IPC, sheet_name=0, header=None, skiprows=3, usecols=[1, 2])
ipc_raw.columns = ["FECHA", "IPC"]
ipc_raw = ipc_raw.dropna(subset=["FECHA"])
ipc_raw["FECHA"] = pd.to_datetime(ipc_raw["FECHA"], errors="coerce")
ipc = ipc_raw.dropna().set_index("FECHA")["IPC"].astype(float)
base_ipc = ipc[ipc.index.year == IPC_BASE_PROM_ANUAL]
IPC_BASE = float(base_ipc.mean()) if len(base_ipc) == 12 else IPC_BASE_FIJO
deflactor = (ipc / IPC_BASE)              # igual a la fila "deflactor" del workbook manual

# Universo de fechas: desde AÑO_INICIO hasta el último mes con algún devengado
fechas_con_datos = pd.concat([df["FECHA"] for df in inputs.values() if not df.empty])
ULTIMO_MES = fechas_con_datos.max()
AÑOS = list(range(AÑO_INICIO, ULTIMO_MES.year + 1))
FECHAS = [datetime(a, m, 1) for a in AÑOS for m in range(1, 13)]

print(f"Series: {AÑO_INICIO}-01 a {ULTIMO_MES:%Y-%m} · IPC base {IPC_BASE_PROM_ANUAL} = {IPC_BASE:.2f} · último IPC {ipc.index.max():%Y-%m}")
if faltantes:
    print("⚠️ Sin input (los bloques quedarán vacíos):", ", ".join(faltantes))
    if any(x.startswith("DEV Deuda") for x in faltantes):
        print("   (sin DEV Deuda el subtotal SERVICIO DE LA DEUDA igual se llena con la genérica 2.8 de DEV Generica; "
              "solo quedan vacías Intereses+comisiones y Amortización en los meses con deuda)")
if not controles.empty:
    print("Controles leídos:", len(controles), "filas ·", (controles["ESTADO"] != "OK").sum(), "con estado distinto de OK")


In [ ]:
# ═══ BLOQUE 4 · CONSTRUCCIÓN DEL WORKBOOK ═══
F_TIT = Font(name="Arial", size=9, bold=True); F_NOR = Font(name="Arial", size=9)
F_SEC = Font(name="Arial", size=9, bold=True); F_SUB = Font(name="Arial", size=9, italic=True)
FILL_H = PatternFill("solid", fgColor="D9E1F2"); FILL_T = PatternFill("solid", fgColor="F2F2F2")
THIN = Side(style="thin", color="999999")

def serie_mensual(df, filtro_cols, valor="DEVENGADO"):
    """-> dict {FECHA: suma} de un DataFrame ya filtrado por departamento."""
    if df is None or df.empty:
        return {}
    for col, vals in filtro_cols.items():
        if vals is not None:
            df = df[df[col].isin(vals)]
    return df.groupby("FECHA")[valor].sum().to_dict()

class Hoja:
    """Escribe una hoja con el formato del workbook manual: fila 4 años, fila 5 fechas,
    columna 'Total' tras cada diciembre, y al final Var.% acumulada Ene.-Mes."""
    def __init__(self, wb, nombre, titulo, cuadro, unidad="(Miles de soles)"):
        self.ws = wb.create_sheet(nombre)
        self.ws["A1"] = cuadro; self.ws["A1"].font = F_TIT
        self.ws["A2"] = titulo; self.ws["A2"].font = F_TIT
        self.ws["A3"] = unidad; self.ws["A3"].font = F_SUB
        self.col_de = {}                 # FECHA -> columna
        self.cols_total = {}             # AÑO -> columna Total
        c = 2
        for a in AÑOS:
            self.ws.cell(4, c, a).font = F_TIT
            for m in range(1, 13):
                f = datetime(a, m, 1)
                cell = self.ws.cell(5, c, f); cell.number_format = "mmm-yy"; cell.font = F_NOR
                cell.alignment = Alignment(horizontal="center")
                self.col_de[f] = c; c += 1
            self.ws.cell(4, c, "Total").font = F_TIT
            self.ws.cell(5, c, a).font = F_TIT
            self.cols_total[a] = c; c += 1
        self.c_fin = c - 1                       # última columna de datos
        self.c_var = c + 1                       # columnas de variación
        self.fila = 7
        self.ws.column_dimensions["A"].width = 46
        for cc in range(2, self.c_var + 6):
            self.ws.column_dimensions[L(cc)].width = 9
        self.ws.freeze_panes = "B6"
        # encabezados del bloque de variación
        u = ULTIMO_MES
        self.ws.cell(4, self.c_var, f"Var.% {u.year}/{u.year-1}").font = F_TIT
        self.ws.cell(5, self.c_var, u.strftime("%b-%y")).font = F_NOR
        self.ws.cell(4, self.c_var + 1, "Var.% acum.").font = F_TIT
        self.ws.cell(5, self.c_var + 1, f"Ene.-{u.strftime('%b.')}").font = F_NOR
        self.ws.cell(4, self.c_var + 2, u.year - 1).font = F_TIT
        self.ws.cell(5, self.c_var + 2, f"Ene.-{u.strftime('%b.')}").font = F_NOR
        self.ws.cell(4, self.c_var + 3, u.year).font = F_TIT
        self.ws.cell(5, self.c_var + 3, f"Ene.-{u.strftime('%b.')}").font = F_NOR
        self.ws.cell(4, self.c_var + 4, "Dif. acum.").font = F_TIT

    def _variacion(self, r):
        """Fórmulas de variación para la fila r (mismo criterio que el workbook manual)."""
        u = ULTIMO_MES
        c_act, c_ant = self.col_de.get(u), self.col_de.get(datetime(u.year - 1, u.month, 1))
        ws = self.ws
        if c_ant:
            ws.cell(r, self.c_var, f'=IFERROR(({L(c_act)}{r}/{L(c_ant)}{r}-1)*100,"")')
        a0, a1 = self.col_de.get(datetime(u.year - 1, 1, 1)), self.col_de.get(datetime(u.year, 1, 1))
        if a0:
            ws.cell(r, self.c_var + 2, f"=SUM({L(a0)}{r}:{L(c_ant)}{r})")
        ws.cell(r, self.c_var + 3, f"=SUM({L(a1)}{r}:{L(c_act)}{r})")
        if a0:
            ws.cell(r, self.c_var + 1, f'=IFERROR(({L(self.c_var+3)}{r}/{L(self.c_var+2)}{r}-1)*100,"")')
            ws.cell(r, self.c_var + 4, f"={L(self.c_var+3)}{r}-{L(self.c_var+2)}{r}")
        for cc in range(self.c_var, self.c_var + 5):
            ws.cell(r, cc).number_format = "#,##0.0"; ws.cell(r, cc).font = F_NOR

    def _totales(self, r):
        for a, c in self.cols_total.items():
            ini, fin = self.col_de[datetime(a, 1, 1)], self.col_de[datetime(a, 12, 1)]
            cell = self.ws.cell(r, c, f"=SUM({L(ini)}{r}:{L(fin)}{r})")
            cell.number_format = "#,##0"; cell.font = F_TIT; cell.fill = FILL_T

    def fila_datos(self, etiqueta, valores, escala=1 / 1000, fmt="#,##0", font=F_NOR, variacion=True, totales=True):
        """valores: dict {FECHA: monto en soles}. Escribe en miles; meses sin dato quedan vacíos."""
        r = self.fila
        self.ws.cell(r, 1, etiqueta).font = font
        for f, c in self.col_de.items():
            if f in valores and f <= ULTIMO_MES:
                cell = self.ws.cell(r, c, round(valores[f] * escala, 3 if escala != 1 / 1000 else 0))
                cell.number_format = fmt; cell.font = font
        if totales:
            self._totales(r)
        if variacion:
            self._variacion(r)
        self.fila += 1
        return r

    def fila_formula(self, etiqueta, plantilla, font=F_SEC, fmt="#,##0", variacion=True, totales=True):
        """plantilla usa {c} como letra de columna, p.ej. '=SUM({c}9:{c}13)'."""
        r = self.fila
        self.ws.cell(r, 1, etiqueta).font = font
        for f, c in self.col_de.items():
            cell = self.ws.cell(r, c, plantilla.format(c=L(c), r=r)); cell.number_format = fmt; cell.font = font
        if totales:
            self._totales(r)
        if variacion:
            self._variacion(r)
        self.fila += 1
        return r

    def fila_ratio(self, etiqueta, fila_num, fila_den, factor=1, font=F_NOR, fmt="#,##0"):
        return self.fila_formula(etiqueta, f"=IFERROR({{c}}{fila_num}/{{c}}{fila_den}{'*' + str(factor) if factor != 1 else ''},\"\")",
                                 font=font, fmt=fmt)

    def texto(self, t, font=F_SUB):
        self.ws.cell(self.fila, 1, t).font = font; self.fila += 1

    def salto(self, n=1):
        self.fila += n

def hoja_gasto(wb, n, cod_depto, nombre_depto, cuadro):
    gen = inputs[(n["corto"], "Generica")]; deu = inputs[(n["corto"], "Deuda")]; fue = inputs[(n["corto"], "Fuente")]
    gen = gen[gen["COD_DEPTO"] == cod_depto] if not gen.empty else gen
    deu = deu[deu["COD_DEPTO"] == cod_depto] if not deu.empty else deu
    fue = fue[fue["COD_DEPTO"] == cod_depto] if not fue.empty else fue
    if not gen.empty:
        gen = gen.assign(CLASE=[clasificar_generica(c, e) for c, e in zip(gen["COD_GENERICA"], gen["GENERICA"])])
    if not fue.empty:
        fue = fue.assign(CLASE=fue["FUENTE"].map(clasificar_fuente))
    # ── CORRECCIÓN servicio de la deuda ──
    # 1) el subtotal SERVICIO DE LA DEUDA sale de la genérica 2.8 (tipo de transacción 7) de DEV Generica, que siempre se
    #    raspa; antes era la suma de Intereses + Amortización y quedaba en 0 cuando el desglose (DEV Deuda) fallaba o faltaba.
    #    Respaldo: la fila TOTAL que ahora guarda el scraper en DEV Deuda.
    # 2) las filas de DEV Deuda con CLASIFICACION = OTRO se reclasifican por el texto de la RUTA.
    # 3) en los meses con dato de genéricas y sin deuda, la deuda se escribe como 0 (no queda en blanco).
    if not deu.empty:
        deu = deu.assign(CLASIFICACION=[c if c in ("TOTAL", "AMORTIZACION", "INTERESES Y COMISIONES") else clasificar_deuda(r)
                                        for c, r in zip(deu["CLASIFICACION"].astype(str), deu["RUTA"].astype(str))])
    tot_deuda = {}
    if not gen.empty:
        es_deuda = gen["COD_GENERICA"].astype(str).str.strip().str.split("-").str[0].eq("7") | gen["GENERICA"].map(lambda e: "servicio de la deuda" in normalizar(e))
        tot_deuda = {f: 0.0 for f in gen["FECHA"].unique()}
        tot_deuda.update(gen[es_deuda].groupby("FECHA")["DEVENGADO"].sum().to_dict())
    elif not deu.empty:
        tot_deuda = serie_mensual(deu, {"CLASIFICACION": ["TOTAL"]})
    h = Hoja(wb, n["hoja_g"], f"{nombre_depto}: GASTOS NO FINANCIEROS DEL {n['nombre']}   1/", cuadro)
    filas = {}
    r_total = h.fila; h.fila += 1                                      # TOTAL DE GASTOS (se llena al final)
    secciones = {}
    seccion = None
    for etiqueta, cods in GENERICAS:
        if cods is None:
            seccion = etiqueta; secciones[seccion] = [h.fila, None]; h.fila += 1
            continue
        if isinstance(cods, str) and cods.startswith("DEUDA:"):
            vals = serie_mensual(deu, {"CLASIFICACION": [cods.split(":", 1)[1]]})
            vals.update({f: 0.0 for f, t in tot_deuda.items() if t == 0 and f not in vals})
        else:
            vals = serie_mensual(gen, {"CLASE": [etiqueta]})
        filas[etiqueta] = h.fila_datos(etiqueta, vals)
        secciones[seccion][1] = filas[etiqueta]
    # subtotales y total (fórmulas, como en el manual)
    for sec, (r, r_fin) in secciones.items():
        r_ini = r + 1
        h.fila, guardado = r, h.fila
        if sec == "SERVICIO DE LA DEUDA" and tot_deuda:
            h.fila_datos(sec, tot_deuda, font=F_SEC)          # total de la genérica 2.8 (ver CORRECCIÓN arriba)
        else:
            h.fila_formula(sec, f"=SUM({{c}}{r_ini}:{{c}}{r_fin})")
        h.fila = guardado
    h.fila, guardado = r_total, h.fila
    h.fila_formula("TOTAL DE GASTOS", "=" + "+".join(f"{{c}}{r}" for r, _ in secciones.values()), font=F_TIT)
    h.fila = guardado
    h.salto()
    r_fuente = h.fila; h.fila += 1
    r_f = []
    for etiqueta, _ in FUENTES:
        r_f.append(h.fila_datos(etiqueta, serie_mensual(fue, {"CLASE": [etiqueta]})))
    h.fila, guardado = r_fuente, h.fila
    h.fila_formula("GASTOS POR FUENTE", f"=SUM({{c}}{r_f[0]}:{{c}}{r_f[-1]})")
    h.fila = guardado
    h.fila_formula("   Diferencia total - fuentes (control)", f"={{c}}{r_total}-{{c}}{r_fuente}", font=F_SUB, variacion=False, totales=False)
    r_deu = secciones["SERVICIO DE LA DEUDA"][0]
    h.fila_formula("   Deuda sin desglosar: total - (intereses + amortización) (control)",
                   f"={{c}}{r_deu}-{{c}}{filas['Intereses+comisiones']}-{{c}}{filas['Amortización']}", font=F_SUB, variacion=False, totales=False)
    h.texto("Fuente: MEF, Consulta Amigable (scraper DEV Generica / DEV Fuente / DEV Deuda)."); h.texto("1/ Preliminar")
    h.salto()
    # ── bloque real ──
    h.texto(f"{nombre_depto}: GASTOS DEL {n['nombre']} EN TÉRMINOS REALES (deflactados con IPC Lima, base {IPC_BASE_PROM_ANUAL} = 100)", F_TIT)
    r_defl = h.fila_datos("Deflactor (IPC / IPC base)", {f: v for f, v in deflactor.items()}, escala=1, fmt="0.0000", variacion=False, totales=False)
    h.fila_ratio("TOTAL DE GASTOS", r_total, r_defl, font=F_TIT)
    for sec, (r, _) in secciones.items():
        h.fila_ratio("  " + sec, r, r_defl, font=F_SEC)
        for etiqueta, cods in GENERICAS:
            if etiqueta in filas and _seccion_de(etiqueta) == sec:
                h.fila_ratio("    " + etiqueta, filas[etiqueta], r_defl)
    h.fila_ratio("GASTOS POR FUENTE DE FINANCIAMIENTO", r_fuente, r_defl, font=F_SEC)
    for (etiqueta, _), r in zip(FUENTES, r_f):
        h.fila_ratio("    " + etiqueta, r, r_defl)
    return h

def _seccion_de(etiqueta):
    sec = None
    for e, c in GENERICAS:
        if c is None:
            sec = e
        if e == etiqueta:
            return sec

def hoja_inversion(wb, n, cod_depto, nombre_depto, cuadro):
    fun = inputs[(n["corto"], "Funcion")]; inv = inputs[(n["corto"], "InvFuente")]
    fun = fun[fun["COD_DEPTO"] == cod_depto] if not fun.empty else fun
    inv = inv[inv["COD_DEPTO"] == cod_depto] if not inv.empty else inv
    if not inv.empty:
        inv = inv.assign(CLASE=inv["FUENTE"].map(clasificar_fuente))
    h = Hoja(wb, n["hoja_i"], f"{nombre_depto}: INVERSIONES NO FINANCIERAS DEL {n['nombre']}   1/", cuadro)
    r_tot = h.fila; h.fila += 2
    r_ini = h.fila
    for f in FUNCIONES:
        cods = [f[:2]]
        if AGRUPAR_COMUNICACIONES_EN_TRANSPORTE:
            if f[:2] == "15":
                cods = ["15", "16"]
            elif f[:2] == "16":
                cods = []
        h.fila_datos(f + (" (incl. en Transporte)" if cods == [] else ""), serie_mensual(fun, {"COD_FUNCION": cods}))
    r_fin = h.fila - 1
    h.fila, g = r_tot, h.fila
    h.fila_formula("INVERSION POR FUNCIONES", f"=SUM({{c}}{r_ini}:{{c}}{r_fin})", font=F_TIT)
    h.fila = g; h.salto()
    r_tf = h.fila; h.fila += 2
    r_fi = h.fila
    for etiqueta, _ in FUENTES:
        h.fila_datos(etiqueta, serie_mensual(inv, {"CLASE": [etiqueta]}))
    r_ff = h.fila - 1
    h.fila, g = r_tf, h.fila
    h.fila_formula("INVERSION POR FUENTE", f"=SUM({{c}}{r_fi}:{{c}}{r_ff})", font=F_TIT)
    h.fila = g
    h.fila_formula("   Diferencia funciones - fuentes (control)", f"={{c}}{r_tot}-{{c}}{r_tf}", font=F_SUB, variacion=False, totales=False)
    h.texto("Fuente: MEF, Consulta Amigable (scraper DEV Funcion / DEV InvFuente). Genérica 2.6 Adquisición de activos no financieros."); h.texto("1/ Preliminar")
    h.salto()
    h.texto(f"{nombre_depto}: INVERSIÓN DEL {n['nombre']} EN TÉRMINOS REALES (IPC Lima, base {IPC_BASE_PROM_ANUAL} = 100)", F_TIT)
    r_defl = h.fila_datos("Deflactor (IPC / IPC base)", {f: v for f, v in deflactor.items()}, escala=1, fmt="0.0000", variacion=False, totales=False)
    h.fila_ratio("INVERSION POR FUNCIONES (real)", r_tot, r_defl, font=F_TIT)
    for i, f in enumerate(FUNCIONES):
        h.fila_ratio("    " + f, r_ini + i, r_defl)
    return h

def hoja_pim(wb, n, cod_depto, nombre_depto):
    """PIM anual por función (todas las genéricas de 2.6 según el scraper PIM) vs devengado 2.6 acumulado."""
    ws = wb.create_sheet(n["hoja_p"])
    ws["A1"] = f"{nombre_depto}: PIM Y AVANCE DE LA INVERSIÓN DEL {n['nombre']} POR FUNCIÓN"; ws["A1"].font = F_TIT
    ws["A2"] = "(Soles; devengado acumulado de la genérica 2.6 al último mes disponible de cada año)"; ws["A2"].font = F_SUB
    ws.column_dimensions["A"].width = 46
    fun = inputs[(n["corto"], "Funcion")]
    fun = fun[fun["COD_DEPTO"] == cod_depto] if not fun.empty else fun
    p = pim[(pim["COD_DEPTO"] == cod_depto) & (pim["NIVEL DE GOBIERNO"].str.upper() == n["pim"])] if not pim.empty else pim
    c = 2
    ws.cell(4, 1, "FUNCIÓN").font = F_TIT
    for a in AÑOS:
        ult = fun[fun["AÑO"] == a]["FECHA"].max() if not fun.empty and (fun["AÑO"] == a).any() else None
        etiqueta_mes = ult.strftime("%b-%y") if ult is not None and pd.notna(ult) else "s/d"
        for j, t in enumerate([f"PIM {a}", f"Devengado a {etiqueta_mes}", "Avance %"]):
            cell = ws.cell(4, c + j, t); cell.font = F_TIT; cell.fill = FILL_H; cell.alignment = Alignment(horizontal="center", wrap_text=True)
            ws.column_dimensions[L(c + j)].width = 14
        for i, f in enumerate(FUNCIONES):
            r = 6 + i
            ws.cell(r, 1, f).font = F_NOR
            v_pim = float(p[p["COD_FUNCION"] == f[:2]]["PIM"].sum()) if not p.empty and (p["AÑO"] == a).any() else None
            if not p.empty:
                v_pim = float(p[(p["COD_FUNCION"] == f[:2]) & (p["AÑO"] == a)]["PIM"].sum())
            v_dev = float(fun[(fun["COD_FUNCION"] == f[:2]) & (fun["AÑO"] == a)]["DEVENGADO"].sum()) if not fun.empty else None
            if v_pim is not None:
                ws.cell(r, c, v_pim).number_format = "#,##0"
            if v_dev is not None:
                ws.cell(r, c + 1, v_dev).number_format = "#,##0"
            ws.cell(r, c + 2, f'=IFERROR({L(c+1)}{r}/{L(c)}{r}*100,"")').number_format = "0.0"
        r = 6 + len(FUNCIONES)
        ws.cell(r, 1, "TOTAL").font = F_TIT
        ws.cell(r, c, f"=SUM({L(c)}6:{L(c)}{r-1})").number_format = "#,##0"
        ws.cell(r, c + 1, f"=SUM({L(c+1)}6:{L(c+1)}{r-1})").number_format = "#,##0"
        ws.cell(r, c + 2, f'=IFERROR({L(c+1)}{r}/{L(c)}{r}*100,"")').number_format = "0.0"
        for cc in range(c, c + 3):
            ws.cell(r, cc).font = F_TIT; ws.cell(r, cc).fill = FILL_T
        c += 4
    ws.cell(r + 2, 1, "Fuente: MEF, Consulta Amigable (scrapers PIM Funcion y DEV Funcion). PIM = presupuesto institucional modificado vigente a la fecha de extracción.").font = F_SUB
    ws.freeze_panes = "B5"

def hoja_control(wb, cod_depto, nombre_depto):
    ws = wb.create_sheet("CONTROL")
    ws["A1"] = f"CONTROLES DE LOS SCRAPERS · {nombre_depto}"; ws["A1"].font = F_TIT
    ws["A2"] = "Resumen de los CSV de control (suma de meses vs vista anual del MEF; PIM por nivel vs total). ESTADO distinto de OK = revisar antes de usar."; ws["A2"].font = F_SUB
    ws["A4"] = f"Generado: {date.today().isoformat()} · último mes con datos: {ULTIMO_MES:%Y-%m}"; ws["A4"].font = F_NOR
    ws.column_dimensions["A"].width = 70; ws.column_dimensions["B"].width = 8
    for j, t in enumerate(["Archivo de control", "Año", "Filas", "OK", "No OK", "Estados", "Extracción"]):
        cell = ws.cell(6, 1 + j, t); cell.font = F_TIT; cell.fill = FILL_H
    if controles.empty:
        ws.cell(7, 1, "No se encontraron CSV de control.").font = F_SUB
        return
    sub = controles[controles["DEPARTAMENTO"].astype(str).str.startswith(cod_depto)]
    r = 7
    for archivo, g in sub.groupby("ARCHIVO"):
        ok = (g["ESTADO"] == "OK").sum()
        ws.cell(r, 1, archivo).font = F_NOR
        ws.cell(r, 2, int(g["AÑO"].iloc[0]) if pd.notna(g["AÑO"].iloc[0]) else "")
        ws.cell(r, 3, len(g)); ws.cell(r, 4, int(ok)); ws.cell(r, 5, int(len(g) - ok))
        ws.cell(r, 6, ", ".join(f"{k}: {v}" for k, v in g["ESTADO"].value_counts().items()))
        ws.cell(r, 7, str(g["FECHA_EXTRACCION"].iloc[0]))
        if ok < len(g):
            for cc in range(1, 8):
                ws.cell(r, cc).font = Font(name="Arial", size=9, bold=True, color="C00000")
        r += 1
    for cc, w in zip("CDEFG", [7, 7, 7, 40, 12]):
        ws.column_dimensions[cc].width = w


In [ ]:
# ═══ BLOQUE 5 · GENERAR UN WORKBOOK POR DEPARTAMENTO ═══
generados = []
for cod, (nombre, archivo) in DEPARTAMENTOS.items():
    wb = Workbook(); wb.remove(wb.active)
    cuadros = iter(range(18, 30))
    for n in NIVELES:
        hoja_gasto(wb, n, cod, nombre, f"CUADRO Nº {next(cuadros)}")
        hoja_inversion(wb, n, cod, nombre, f"CUADRO Nº {next(cuadros)}")
    for n in NIVELES:
        hoja_pim(wb, n, cod, nombre)
    hoja_control(wb, cod, nombre)
    salida = os.path.join(RUTA_SALIDA, f"Fiscal-{archivo}{SUFIJO_SALIDA}.xlsx")
    wb.save(salida); generados.append(salida)
    print("💾", salida)

print("\nResumen de inputs usados (filas por nivel y dimensión):")
display(pd.DataFrame([{"nivel": k[0], "dimensión": k[1], "filas": len(v), "desde": (v["FECHA"].min().strftime("%Y-%m") if not v.empty else "-"),
                       "hasta": (v["FECHA"].max().strftime("%Y-%m") if not v.empty else "-")} for k, v in inputs.items()]))
if faltantes:
    print("⚠️ Bloques vacíos por falta de input:", ", ".join(faltantes))
